# Build a model-ready dataset for house price prediction\n\nStarts from `merged_listings_rera.csv` (output of `02_merge.ipynb`) and produces a cleaned, feature-engineered table with `Price_INR` as the regression target. Scope is deliberately **sale listings only** — rent/lease listings are a different price regime (deposits and monthly rents overlap numerically with sale prices) and are dropped rather than mixed in.

In [1]:
import pandas as pd
import numpy as np

RAW = "../raw"
df = pd.read_csv(f"{RAW}/merged_listings_rera.csv", low_memory=False)
df.shape

(93621, 41)

## Deal type: keep sale listings only

In [2]:
url = df["URL"].astype(str).str.lower()
is_rent = url.str.contains("for-rent|-rent-|for-lease", regex=True)
is_sale = url.str.contains("for-sale|resale|-sale-", regex=True) & ~is_rent

df["deal_type"] = np.where(is_sale, "Sale", np.where(is_rent, "Rent", "Unknown"))
df["deal_type"].value_counts()

deal_type
Sale       53069
Unknown    38636
Rent        1916
Name: count, dtype: int64

In [3]:
df = df[df["deal_type"] == "Sale"].drop_duplicates(subset="URL").copy()
df.shape

(34821, 42)

## Drop unusable / redundant columns

In [4]:
drop_cols = [
    "URL", "_links", "deal_type", "Transaction", "YearMonth", "Year", "Month",
    "Pincode", "Full_Address", "match_score", "match_method", "rera_token",
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
df.shape

(34821, 31)

## Outlier filtering (1st–99th percentile on price & area)

In [5]:
for col in ["Price_INR", "Area_Sqft", "PricePerSqft"]:
    lo, hi = df[col].quantile([.01, .99])
    df = df[df[col].between(lo, hi)]
df.shape

(32158, 31)

## Missing values

In [6]:
numeric_cols = ["BHK", "Area_Sqft", "Bathrooms", "Balconies", "Car_Parking",
                "Floor_Number", "Total_Floors", "Property_Age_Years",
                "Price_INR", "PricePerSqft", "Latitude", "Longitude",
                "Cement_Price_Rs_per_bag_50kg", "Steel_TMT_Price_Rs_per_tonne"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

zero_fill = ["Balconies", "Car_Parking", "Bathrooms"]
for col in zero_fill:
    df[col] = df[col].fillna(0)

median_fill = ["Floor_Number", "Total_Floors", "Property_Age_Years", "BHK"]
for col in median_fill:
    df[col] = df[col].fillna(df[col].median())

unknown_fill = ["Furnishing", "Facing", "Ownership", "Overlooking", "Age_Group",
                "Construction_Status", "Society", "Locality"]
for col in unknown_fill:
    df[col] = df[col].fillna("Unknown")

df.isna().mean().sort_values(ascending=False).head(10)

rera_approved_date               0.841346
rera_project_status              0.837086
rera_district                    0.837086
rera_promoter_name               0.837086
rera_proposed_completion_date    0.837086
rera_pincode                     0.837086
rera_project_name                0.837086
rera_project_type                0.837086
Price_INR                        0.000000
Price_Cr                         0.000000
dtype: float64

## Feature engineering

In [7]:
df["log_price"] = np.log1p(df["Price_INR"])
df["floor_ratio"] = np.where(df["Total_Floors"] > 0, df["Floor_Number"] / df["Total_Floors"], 0)
df["floor_ratio"] = df["floor_ratio"].clip(0, 1)
df["has_rera_match"] = df["rera_project_name"].notna().astype(int)

for col in ["rera_project_status", "rera_project_type", "rera_district"]:
    df[col] = df[col].fillna("Not Matched")

df[["log_price", "floor_ratio", "has_rera_match"]].describe()

,log_price,floor_ratio,has_rera_match
count,32158.000000,32158.000000,32158.000000
mean,16.245127,0.556844,0.162914
std,0.863915,0.273333,0.369293
min,14.077876,0.000000,0.000000
25%,15.640060,0.400000,0.000000
50%,16.185754,0.600000,0.000000
75%,16.811243,0.666667,0.000000
max,18.683045,1.000000,1.000000


## Checkpoint: save the pre-one-hot cleaned table\n\nSaved separately from the one-hot feature matrix below -- `04_in_depth_eda.ipynb` and `05_preprocessing.ipynb` both work with plain category columns (e.g. `Locality`, `Construction_Status`), not one-hot dummies, so they read this checkpoint rather than the final matrix.

In [8]:
df.to_csv(f"{RAW}/sale_only_model_cleaned_listings.csv", index=False)
print(f"checkpoint: {len(df):,} rows x {df.shape[1]} columns -> {RAW}/sale_only_model_cleaned_listings.csv")

checkpoint: 32,158 rows x 34 columns -> ../raw/sale_only_model_cleaned_listings.csv


## Collapse noisy/free-text categoricals before one-hot encoding\n\nSeveral fields (`Overlooking`, `Furnishing`, `Construction_Status`, `Ownership`) carry scrape artifacts — breadcrumb text, stray placeholder values — inflating what should be a handful of categories into hundreds or thousands. Keep each column's top 10 values by frequency; fold everything else into `"Other"`.

In [9]:
low_card_cats = ["Furnishing", "Facing", "Ownership", "Overlooking", "Age_Group",
                  "Construction_Status", "rera_project_status", "rera_project_type"]

for col in low_card_cats:
    top = df[col].value_counts().head(10).index
    df[col] = df[col].where(df[col].isin(top), "Other")

df = pd.get_dummies(df, columns=low_card_cats, prefix=low_card_cats)
df.shape

(32158, 95)

**High-cardinality columns kept as plain strings** (`Locality`, `Society`, `rera_district`, `rera_promoter_name`, `rera_approved_date`, `rera_proposed_completion_date`): too many unique values for one-hot encoding — use frequency or target encoding at model-training time.

In [10]:
out_path = f"{RAW}/sale_only_model_ready_listings.csv"
df.to_csv(out_path, index=False)
print(f"wrote {len(df):,} rows x {df.shape[1]} columns -> {out_path}")
df.describe(include="all").T.head(20)

wrote 32,158 rows x 95 columns -> ../raw/sale_only_model_ready_listings.csv


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Price_INR,32158.0,NaN,NaN,NaN,16771765.18123,17534815.955961,1300000.0,6200000.0,10700000.0,20000000.0,130000000.0
Price_Cr,32158.0,NaN,NaN,NaN,1.677177,1.753481,0.13,0.62,1.07,2.0,13.0
PricePerSqft,32158.0,NaN,NaN,NaN,9294.264069,9196.874529,2569.0,5333.0,7024.0,10236.22,109000.0
BHK,32158.0,NaN,NaN,NaN,2.708595,0.878689,1.0,2.0,3.0,3.0,10.0
Area_Sqft,32158.0,NaN,NaN,NaN,1790.012199,1234.172389,133.33,1050.0,1413.0,2045.0,7943.0
Bathrooms,32158.0,NaN,NaN,NaN,2.947758,1.921948,0.0,2.0,2.0,3.0,44.0
Balconies,32158.0,NaN,NaN,NaN,0.836402,1.12872,0.0,0.0,0.0,2.0,16.0
Car_Parking,32158.0,NaN,NaN,NaN,0.815909,1.330253,0.0,1.0,1.0,1.0,203.0
Floor_Number,32158.0,NaN,NaN,NaN,6.675788,46.020272,0.0,3.0,3.0,4.0,2802.0
Total_Floors,32158.0,NaN,NaN,NaN,3451717.001399,618982631.951294,0.0,5.0,5.0,5.0,111000000000.0
